# Gender classification on Simpsons dialogue (Multinomial Naive Bayes)

This notebook reproduces `gender-convo-simpsonscorpus-MNB.py` (week3) but reads the raw Simpsons data files directly instead of using `convokit`.

It expects the following files to already be available locally in the Colab runtime, under `/content/sample_data/` (the standard Colab `sample_data` folder, visible in the Files pane):
- `simpsons_speakers.txt` (columns: `speaker_id`, `gender`)
- `simpsons_lines.txt` (columns: `line_id`, `speaker_id`, `text`)

Drag these two files directly into the `sample_data` folder in the Colab Files pane before running the cells below.

We then:
1. Split utterances into texts spoken by male vs. female characters.
2. Vectorize the texts with TF-IDF.
3. Train a Multinomial Naive Bayes classifier to predict speaker gender from the text.
4. Inspect the most salient words per class and evaluate precision, recall and accuracy.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.naive_bayes import MultinomialNB

## 1. Load the data

We read the speakers file to build a `speaker_id -> gender` lookup, then read the lines file and split the utterance texts into `male` and `female` lists based on the speaker's gender. This replaces the `convokit.Corpus` used in the original script.

In [ ]:
SPEAKERS_PATH = "/content/sample_data/simpsons_speakers.txt"
LINES_PATH = "/content/sample_data/simpsons_lines.txt"


def load_speaker_genders(path):
    genders = {}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 2:
                continue
            speaker_id, gender = fields[0], fields[1]
            genders[speaker_id] = gender
    return genders


def process_data():
    print("Loading data...")
    genders = load_speaker_genders(SPEAKERS_PATH)

    male = []
    female = []
    with open(LINES_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 3:
                continue
            _line_id, speaker_id, text = fields[0], fields[1], fields[2]
            gender = genders.get(speaker_id, "?")
            if gender == "f":
                female.append(text)
            elif gender == "m":
                male.append(text)
    print("Done.")
    print("No. male:", len(male), "No. female:", len(female))
    return male[:1000], female[:1000]


male, female = process_data()

## 2. Vectorize and prepare train/test splits

Male utterances are labeled `0.0`, female utterances `1.0`. We fit a TF-IDF vectorizer on all utterances, then vectorize the train/test splits separately, as in the original script.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
vectorizer.fit(male + female)

labeled_utt = [(m, 0.0) for m in male] + [(f, 1.0) for f in female]
labels = [l for (_utt, l) in labeled_utt]

labeled_utt_train, labeled_utt_test, labels_train, labels_test = train_test_split(
    labeled_utt, labels, test_size=0.1
)

X_train = []
X_test = []
y_train = []
y_test = []
text_test = []

for utt, label in labeled_utt_train:
    values = list(vectorizer.transform([utt]).toarray()[0])
    X_train.append(values)
    y_train.append(label)

for utt, label in labeled_utt_test:
    values = list(vectorizer.transform([utt]).toarray()[0])
    X_test.append(values)
    y_test.append(label)
    text_test.append(utt)

## 3. Train the Multinomial Naive Bayes classifier

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

## 4. Most salient words per class

Rank the vocabulary by log-probability for each class (0 = male, 1 = female) and save the top-100 words per class.

In [ ]:
def get_salient_words(model, vectorizer, class_id):
    words = vectorizer.get_feature_names_out()
    zipped = list(zip(words, model.feature_log_prob_[class_id]))
    sorted_zip = sorted(zipped, key=lambda t: t[1], reverse=True)
    return sorted_zip


with open("simpsons-salient-words-100.txt", "w") as fp_w:
    fp_w.write("Male:" + str(get_salient_words(clf, vectorizer, 0)[:100]) + "\n\n")
    fp_w.write("Female:" + str(get_salient_words(clf, vectorizer, 1)[:100]))
print("See simpsons-salient-words-100.txt for top-100 most salient words per class.")

## 5. Evaluate on the test set

In [ ]:
y_pred = clf.predict(X_test)
n = 0
for pred in y_pred:
    print(text_test[n], y_test[n], pred)
    n += 1

print("Precision:", metrics.precision_score(y_test, y_pred))
print("Recall:", metrics.recall_score(y_test, y_pred))
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))